# Module 5 — Notebook 05: Martin LIANA (full-circuit replication)

Module 1's circuit readout, repeated on Martin: ligand–receptor interaction
strength per condition (Involved vs Uninvolved), grouped by compartment.
The pairs that decide replication:

- OSM -> OSMR (and OSM -> LIFR): off/low in Uninvolved, on in Involved?
- IL11 -> IL11RA: Involved-only?
- IL1B -> SIGIRR: the brake — stronger in Uninvolved, weaker in Involved
  (the mirror image of Smillie's Healthy > Inflamed gradient)?

Compartment labels come from notebook 03's marker dotplot (verified there).
The LIANA receptor-complex naming fix from module 1 notebook 04 is baked in
(substring matching: OSMR appears as IL6ST_OSMR, IL11RA as IL11RA_IL6ST).

Analyses of public datasets; all data credited to original authors.


In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li

base = os.path.expanduser("~/Documents/immune-endotypes/data/martin_2019")
adata = sc.read_h5ad(os.path.join(base, "adata_martin_qc.h5ad"))
COMP = {"0": "T", "1": "T", "8": "T", "9": "T", "2": "B", "3": "Plasma", "7": "Plasma",
        "12": "Plasma", "13": "Plasma", "4": "Myeloid", "5": "Epithelial", "6": "Epithelial",
        "10": "Fibroblast", "11": "Endothelial", "14": "Other"}
adata.obs["compartment"] = adata.obs["leiden_0.5"].astype(str).map(COMP)
print(adata.obs.groupby(["compartment", "condition"]).size().unstack(fill_value=0))


In [ ]:
# Label-stratified subsample (LIANA cost control, as in module 1 nb04)
rng = np.random.default_rng(0)
keep = []
for (cond, comp), idx in adata.obs.groupby(["condition", "compartment"]).groups.items():
    idx = np.asarray(idx)
    keep.extend(rng.choice(idx, size=min(1000, len(idx)), replace=False) if len(idx) > 1000 else idx)
sub = adata[keep].copy()
print("subsample:", sub.shape)


In [ ]:
res = {}
for cond in ["Uninvolved", "Involved"]:
    a = sub[sub.obs["condition"] == cond].copy()
    li.mt.rank_aggregate(a, groupby="compartment", expr_prop=0.1,
                         resource_name="consensus", n_perms=100, use_raw=False,
                         key_added=f"liana_{cond}", verbose=False)
    r = a.uns[f"liana_{cond}"].copy()
    r["condition"] = cond
    res[cond] = r
    r.to_csv(f"../results/liana_martin_{cond}.csv", index=False)
    print(cond, r.shape)


In [ ]:
# Watch pairs: ligand exact, receptor substring (complex-aware)
WATCH = [("OSM", "OSMR"), ("OSM", "LIFR"), ("IL11", "IL11RA"), ("IL1B", "SIGIRR"),
         ("IL1B", "IL1R1"), ("TNF", "TNFRSF1A"), ("CCL2", "CCR2"), ("IL6", "IL6R")]
rows = []
for cond, r in res.items():
    for lig, rec in WATCH:
        m = r[(r["ligand_complex"] == lig) & (r["receptor_complex"].str.contains(rec, regex=False))]
        if len(m):
            best = m.loc[m["magnitude_rank"].idxmin()]
            rows.append({"pair": f"{lig}-{rec}", "condition": cond,
                         "source": best["source"], "target": best["target"],
                         "strength_1_over_rank": round(1.0 / best["magnitude_rank"], 2)})
        else:
            rows.append({"pair": f"{lig}-{rec}", "condition": cond,
                         "source": "-", "target": "-", "strength_1_over_rank": 0.0})
w = pd.DataFrame(rows)
print(w.pivot_table(index="pair", columns="condition", values="strength_1_over_rank").fillna(0))
print()
print(w.sort_values(["pair", "condition"]).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
piv = w.pivot_table(index="pair", columns="condition", values="strength_1_over_rank").fillna(0)
ax = piv.plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("interaction strength (1 / magnitude rank)")
ax.set_title("Martin: circuit watch pairs by condition")
plt.tight_layout(); plt.savefig("../figures/martin_liana_watch.png", dpi=130); plt.show()


## Read-out (fill in after running)

- OSM-OSMR: Uninvolved vs Involved strength — the switch replicated?
- IL11-IL11RA: Involved-only, as in Smillie?
- IL1B-SIGIRR: stronger in Uninvolved (brake on) than Involved (brake off)?
- Next: notebook 06 — human integration (Smillie + Martin + Kinchen);
  notebook 07 — Kinchen mouse arm, cross-species scoring; then the
  cross-cohort summary figure with exact numbers for the manuscript draft.
